# ORACLE — ACDC segmentation → ONNX INT8 export (PRD-002 §14)

**Human-run on Kaggle GPU** (T4). Produces Kaggle dataset `oracle-seg-v1` with
`seg.onnx`, `seg_int8.onnx`, `meta.json`, `metrics.csv`.

> B-009 finding (validated): the HF model `MohidAbdullah/ACDC-Heart-Segmentation`
> uses **min-max [0,1] normalization** (z-score degrades MYO Dice 0.889→0.495).
> The exported `meta.json` therefore carries `"norm": "minmax01"` and native
> class order `["BG","RV","MYO","LV"]` (matches PRD §6).

In [ ]:
!pip install -q torch==2.3.* onnx>=1.16 onnxruntime numpy scipy huggingface_hub

In [ ]:
# Cell 2 — dataset mount (ACDC official challenge mirror; verify slug at runtime)
from kaggle_datasets import KaggleDatasets
import os, glob
ACDC_DIR = os.environ.get("ACDC_DIR")
if not ACDC_DIR:
    ACDC_DIR = KaggleDatasets().get_gcs_path("acdc-cardiac-mri")  # verify slug!
patients = sorted(glob.glob(f"{ACDC_DIR}/*/"))
print("ACDC root:", ACDC_DIR, "| patients:", len(patients))
# Fallback CT path (HF): AI-CVM/Cardiac-CT

In [ ]:
# Cell 3 — load HF model (MIT): MohidAbdullah/ACDC-Heart-Segmentation
import torch
from huggingface_hub import PyTorchModelHubMixin

class ACDCUNet(torch.nn.Module):
    # strict contract verified in B-009: strict load 160/160 tensors
    ...

model = ACDCUNet.from_pretrained("MohidAbdullah/ACDC-Heart-Segmentation")
model.eval()
print(sum(p.numel() for p in model.parameters())/1e6, "M params")

In [ ]:
# Cell 4 — eval Dice/HD95 on the ACDC test split → metrics.csv
# Preprocessing (B-009): per-slice min-max [0,1]; class order ["BG","RV","MYO","LV"]
import numpy as np, pandas as pd
from oracle_style.metrics import dice, hd95  # inline copies below

def metrics_csv(dice_lv, dice_myo, dice_rv, hd95_lv):
    df = pd.DataFrame([{
        "dice_LV": dice_lv, "dice_MY0": dice_myo, "dice_RV": dice_rv, "hd95_LV": hd95_lv,
        "thresholds": "LV>=0.90 MYO>=0.82 RV>=0.85 HD95(LV)<=6mm",
    }])
    df.to_csv("metrics.csv", index=False)
    print(df)
# PRD P2 gates: LV>=0.90, MYO>=0.82, RV>=0.85, HD95(LV)<=6mm

In [ ]:
# Cell 5 — torch.onnx.export (opset 17, dynamic batch)
dummy = torch.randn(1, 1, 256, 256)
torch.onnx.export(
    model, dummy, "seg.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "N"}, "output": {0: "N"}},
    opset_version=17,
)
print("seg.onnx written")

In [ ]:
# Cell 6 — dynamic INT8 quantization
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic("seg.onnx", "seg_int8.onnx", weight_type=QuantType.QInt8)
import os
print("fp32:", os.path.getsize("seg.onnx")//1024, "KB | int8:", os.path.getsize("seg_int8.onnx")//1024, "KB")

In [ ]:
# Cell 7 — meta.json sidecar (PRD §6 contract + B-009 norm field)
import json
meta = {
    "input_name": "input", "output_name": "output", "input_size": 256,
    "classes": ["BG", "RV", "MYO", "LV"],
    "mean": 0.0, "std": 1.0, "opset": 17,
    "source": "hf:MohidAbdullah/ACDC-Heart-Segmentation",
    "quant": "int8-dynamic",
    "norm": "minmax01",
}
json.dump(meta, open("meta.json", "w"), indent=2)
print(json.dumps(meta, indent=2))

In [ ]:
# Cell 8 — CPU timing cell (2 threads, emulate i5-6500 slice path)
import onnxruntime as ort, time, numpy as np
so = ort.SessionOptions(); so.intra_op_num_threads = 2
sess = ort.InferenceSession("seg_int8.onnx", so, providers=["CPUExecutionProvider"])
x = np.random.rand(1, 1, 256, 256).astype(np.float32)
t = [ (lambda s=time.perf_counter(): (sess.run(None, {"input": x}), time.perf_counter()-s)[1])() for _ in range(20) ]
print(f"median {np.median(t)*1000:.1f} ms/slice @2 threads")

In [ ]:
# Cell 9 — save all artifacts as Kaggle dataset version oracle-seg-v1
import json, os
os.makedirs("oracle-seg-v1", exist_ok=True)
for f in ["seg.onnx", "seg_int8.onnx", "meta.json", "metrics.csv"]:
    os.replace(f, f"oracle-seg-v1/{f}")
json.dump({"title": "oracle-seg-v1", "id": f"{os.environ.get('KAGGLE_USERNAME','YOUR-KAGGLE-USER')}/oracle-seg-v1"},
          open("oracle-seg-v1/dataset-metadata.json", "w"), indent=2)
print(sorted(os.listdir("oracle-seg-v1")))

In [ ]:
# Cell 10 — local pull command
print("kaggle datasets download -d $KAGGLE_USERNAME/oracle-seg-v1 -p models/ --unzip")